In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv("../data/train.csv")
test  = pd.read_csv("../data/test.csv")

In [33]:
train['var3'].replace(-999999, 2, inplace=True)
test['var3'].replace(-999999, 2, inplace=True)

y = train['TARGET']
X = train.drop(['ID', 'TARGET'], axis=1)

X_test_only = test.drop(['ID'], axis=1)

In [34]:
# df.info()

# print("\n 결측값의 수:", df.isna().sum().sum())

# <class 'pandas.core.frame.DataFrame'>
# RangeIndex: 76020 entries, 0 to 76019
# Columns: 369 entries, var3 to var38
# dtypes: float64(111), int64(258)
# memory usage: 214.0 MB

#  결측값의 수: 0

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)

Train shape: (60816, 369)
Validation shape: (15204, 369)


In [60]:
# -----------------------------
# 4. StandardScaler
# -----------------------------
scaler = StandardScaler()
scaler.fit(X_train)          # ✔ train만 fit

X_train_scaled = scaler.transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test_only)   # 여기서 test도 transform만!

In [61]:
# 레이블의 분포 확인
cust_cnt = y.value_counts()
print(cust_cnt) # 1이 불만족 3008명, 만족이 73012

# 불만족고객의 비율
cust_rate = cust_cnt[1] / cust_cnt.sum()
print(f'불만족 고객 비율: {cust_rate:.2f}')

TARGET
0    73012
1     3008
Name: count, dtype: int64
불만족 고객 비율: 0.04


In [64]:
# -----------------------------
# 5. 모델 학습: XGBoost
# -----------------------------

# xgb = XGBClassifier(
#     n_estimators=300,
#     learning_rate=0.05,
#     max_depth=5,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     eval_metric='logloss',
#     random_state=42
# )

# xgb.fit(X_train_scaled, y_train)
# xgb_pred = xgb.predict(X_test_scaled)
# xgb_prob = xgb.predict_proba(X_test_scaled)[:, 1]

# print("\n===== XGBoost =====")
# print("Accuracy :", accuracy_score(y_test, xgb_pred))
# print("F1 Score :", f1_score(y_test, xgb_pred))
# print("ROC-AUC  :", roc_auc_score(y_test, xgb_prob))

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)

xgb.fit(X_train_scaled, y_train)
pred_val = xgb.predict(X_val_scaled)
proba_val = xgb.predict_proba(X_val_scaled)[:,1]

# thresholds = np.arange(0.01, 0.50, 0.01)
# f1_scores = []

# for t in thresholds:
#     pred_t = (proba_val >= t).astype(int)
#     f1_scores.append(f1_score(y_val, pred_t))

# best_idx = np.argmax(f1_scores)
# best_threshold = thresholds[best_idx]
# best_f1 = f1_scores[best_idx]

# print("\n===== Best Threshold =====")
# print("Best Threshold:", best_threshold)
# print("Best F1 Score:", best_f1)

# val_pred_best = (proba_val >= best_threshold).astype(int)

acc = accuracy_score(y_val, pred_val)
f1  = f1_score(y_val, pred_val)
auc = roc_auc_score(y_val, proba_val)

print("\n===== XGBoost =====")
print(f"Accuracy :{acc:.4f}")
print(f"F1 Score :{f1:.4f}")
print(f"ROC-AUC  :{auc:.4f}")



===== XGBoost =====
Accuracy :0.9604
F1 Score :0.0099
ROC-AUC  :0.8509


In [ ]:
import sys
import os

# SantanderCS 폴더를 프로젝트 루트로 설정
current_dir = os.getcwd()  # src 폴더
project_root = os.path.dirname(current_dir)  # SantanderCS 폴더

from utils import get_clf_eval

In [62]:
# -----------------------------
# 6. RandomForest
# -----------------------------

# rf = RandomForestClassifier(
#     n_estimators=300,
#     max_depth=None,
#     random_state=42,
#     n_jobs=-1
# )

# rf.fit(X_train, y_train)      # 랜포는 스케일링 필요 없음
# rf_pred = rf.predict(X_test)
# rf_prob = rf.predict_proba(X_test)[:, 1]

# print("\n===== RandomForest =====")
# print("Accuracy :", accuracy_score(y_test, rf_pred))
# print("F1 Score :", f1_score(y_test, rf_pred))
# print("ROC-AUC  :", roc_auc_score(y_test, rf_prob))

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=0,
    n_jobs=-1,
    max_depth=8
)

rf.fit(X_train, y_train)    # 랜포는 스케일링 필요 없음
rf_pred = rf.predict(X_val)
rf_proba = rf.predict_proba(X_val)[:, 1]

# f1_scores_rf = []

# for t in thresholds:
#     pred_t = (rf_proba >= t).astype(int)
#     f1_scores_rf.append(f1_score(y_val, pred_t))

# best_idx_rf = np.argmax(f1_scores_rf)
# best_threshold_rf = thresholds[best_idx_rf]
# best_f1_rf = f1_scores_rf[best_idx_rf]

# print("\n===== RandomForest Best Threshold =====")
# print(f"Best Threshold :{best_threshold_rf:.4f}")
# print(f"Best F1 Score  :{best_f1_rf:.4f}")

# # 최적 threshold 적용
# val_pred_best_rf = (rf_proba >= best_threshold_rf).astype(int)

# 성능 측정
acc_best_rf = accuracy_score(y_val, rf_pred)
f1_best_rf  = f1_score(y_val, rf_pred)
auc_best_rf = roc_auc_score(y_val, rf_proba)

print("\n===== RandomForest =====")
# print(f"Threshold :{best_threshold_rf:.4f}")
print(f"Accuracy  :{acc_best_rf:.4f}")
print(f"F1 Score  :{f1_best_rf:.4f}")
print(f"ROC-AUC   :{auc_best_rf:.4f}")



===== RandomForest =====
Accuracy  :0.9604
F1 Score  :0.0000
ROC-AUC   :0.8131


In [ ]:
# ============================================
# 6. TEST.CSV에 대한 최종 예측
#
# TARGET=1(불만족)을 얼마나 잘 잡아내는지
# 즉, “잠재적으로 문제가 생길 고객”을 잘 찾아내는지?
#
# 불만족(=1) 고객 비율이 너무 낮기 때문에
# F1 score과 recall이 낮게 나오는 것이 정상이고,
# AUC를 중심으로 보는 게 맞아.
# ============================================

# XGBoost test 예측값
xgb_test_pred = xgb.predict_proba(X_test_scaled)[:, 1]

# RandomForest test 예측값
rf_test_pred = rf.predict_proba(X_test_only)[:, 1]

print("\n===== FINAL TEST PREDICTIONS =====")
print("\nXGBoost Test Predictions (probability of TARGET=1(불만족)):")
print(xgb_test_pred[:20])   # 상위 20개만 미리보기

print("\nRandomForest Test Predictions (probability of TARGET=1(불만족)):")
print(rf_test_pred[:20])    # 상위 20개만 미리보기


===== FINAL TEST PREDICTIONS =====

XGBoost Test Predictions (probability of TARGET=1(불만족)):
[0.04930619 0.05511775 0.00103434 0.00724789 0.00128104 0.25067496
 0.01753684 0.18530677 0.02739658 0.01895448 0.02584288 0.00335761
 0.01247218 0.01089097 0.00569734 0.02837694 0.1344037  0.00233657
 0.01139107 0.01790423]

RandomForest Test Predictions (probability of TARGET=1(불만족)):
[0.14       0.00333333 0.         0.         0.         0.2
 0.07333333 0.01666667 0.01666667 0.00666667 0.         0.
 0.02333333 0.00333333 0.00333333 0.04666667 0.11666667 0.
 0.00444444 0.02666667]
